# Scientific Computing with SciPy

SciPy builds on NumPy and provides the algorithms that show up across science, engineering, and data analysis: statistical distributions, numerical integration, optimisation, interpolation, and linear algebra. Where NumPy gives you the array, SciPy gives you the mathematics to do something meaningful with it.

**What's inside:** probability distributions and statistical tests, root finding and minimisation, curve fitting, numerical integration and ODEs, interpolation, and linear algebra.

**Learn more:** [SciPy documentation](https://docs.scipy.org/doc/scipy/)

## Setup

In [ ]:
%pip install scipy numpy

## 1. Statistics (scipy.stats)

Ref: https://docs.scipy.org/doc/scipy/reference/stats.html

### 1.1 Probability distributions

Every distribution object has the same interface: `pdf`, `cdf`, `ppf` (inverse CDF), and `rvs` (random samples).

In [ ]:
from scipy import stats
import numpy as np

dist = stats.norm(loc=0, scale=1)   # standard normal

print(dist.pdf(0).round(4))    # probability density at 0
print(dist.cdf(1.96).round(4)) # P(X <= 1.96)
print(dist.ppf(0.975).round(4))# value where P(X <= x) = 0.975

In [ ]:
# generate random samples from any distribution
rng = np.random.default_rng(42)
samples = dist.rvs(size=1000, random_state=rng)
samples[:5].round(4)

In [ ]:
# other common distributions work the same way
print(stats.poisson(mu=3).pmf(2).round(4))       # P(X = 2) for Poisson(3)
print(stats.binom(n=10, p=0.5).cdf(4).round(4))  # P(X <= 4) for Binomial(10, 0.5)
print(stats.expon(scale=2).mean())                # mean of Exponential(scale=2)

### 1.2 Descriptive statistics

In [ ]:
data = rng.normal(loc=5, scale=2, size=200)

result = stats.describe(data)
print(f'n={result.nobs}')
print(f'mean={result.mean:.4f}')
print(f'variance={result.variance:.4f}')
print(f'skewness={result.skewness:.4f}')
print(f'kurtosis={result.kurtosis:.4f}')

In [ ]:
# percentiles and trimmed mean
print(np.percentile(data, [25, 50, 75]).round(4))
print(stats.trim_mean(data, 0.1).round(4))   # mean after trimming 10% from each tail

### 1.3 Statistical tests

All test functions return a result object with a `statistic` and a `pvalue`.

In [ ]:
# one-sample t-test: is the mean significantly different from 5?
result = stats.ttest_1samp(data, popmean=5)
print(f'statistic={result.statistic:.4f}')
print(f'p-value={result.pvalue:.4f}')

In [ ]:
# two-sample t-test: are two groups different?
a = rng.normal(5, 1, 50)
b = rng.normal(5.5, 1, 50)
result = stats.ttest_ind(a, b)
print(f'p-value={result.pvalue:.4f}')   # small p → groups are likely different

In [ ]:
# normality test: does the data follow a normal distribution?
result = stats.shapiro(data[:50])   # Shapiro-Wilk (best for n < 2000)
print(f'p-value={result.pvalue:.4f}')   # large p → consistent with normal

In [ ]:
# correlation and p-value
x = rng.normal(size=50)
y = x * 0.8 + rng.normal(scale=0.5, size=50)
r, p = stats.pearsonr(x, y)
print(f'r={r:.4f}, p={p:.4f}')

## 2. Optimisation (scipy.optimize)

Ref: https://docs.scipy.org/doc/scipy/reference/optimize.html

### 2.1 Root finding

`fsolve` finds where a function equals zero.

In [ ]:
from scipy.optimize import fsolve

# find x where x^3 - x - 2 = 0
def f(x):
    return x**3 - x - 2

root = fsolve(f, x0=1.5)   # x0 is the starting guess
print(root)
print(f(root).round(10))   # verify: should be ~0

In [ ]:
# brentq finds a root within a bracket [a, b] (more robust than fsolve)
from scipy.optimize import brentq

root = brentq(f, a=1, b=2)   # root must lie between 1 and 2
print(round(root, 6))

### 2.2 Minimisation

In [ ]:
from scipy.optimize import minimize

# find x that minimises (x - 3)^2 + 1
result = minimize(lambda x: (x - 3)**2 + 1, x0=0)
print(f'minimum at x={result.x[0]:.4f}')
print(f'minimum value={result.fun:.4f}')

In [ ]:
# multi-variable minimisation: Rosenbrock function
def rosenbrock(xy):
    x, y = xy
    return (1 - x)**2 + 100 * (y - x**2)**2

result = minimize(rosenbrock, x0=[0, 0], method='BFGS')
print(f'minimum at ({result.x[0]:.4f}, {result.x[1]:.4f})')  # known minimum: (1, 1)

### 2.3 Curve fitting

`curve_fit` finds the parameters that best fit a model function to data.

In [ ]:
from scipy.optimize import curve_fit

# generate noisy data from a known model: y = a * exp(-b * x)
x = np.linspace(0, 4, 50)
true_a, true_b = 3.0, 0.5
y_true = true_a * np.exp(-true_b * x)
y_noisy = y_true + rng.normal(scale=0.1, size=len(x))

# fit the model to the noisy data
def model(x, a, b):
    return a * np.exp(-b * x)

popt, pcov = curve_fit(model, x, y_noisy)
print(f'fitted a={popt[0]:.4f} (true={true_a})')
print(f'fitted b={popt[1]:.4f} (true={true_b})')

In [ ]:
# standard errors of the fitted parameters
perr = np.sqrt(np.diag(pcov))
print(f'std error a: {perr[0]:.4f}')
print(f'std error b: {perr[1]:.4f}')

## 3. Integration (scipy.integrate)

Ref: https://docs.scipy.org/doc/scipy/reference/integrate.html

### 3.1 Numerical integration

`quad` computes a definite integral and returns the result and an error estimate.

In [ ]:
from scipy.integrate import quad

# integrate x^2 from 0 to 3 (exact answer: 9)
result, error = quad(lambda x: x**2, 0, 3)
print(f'result={result:.6f}')
print(f'error estimate={error:.2e}')

In [ ]:
# integrate a normal PDF from -inf to 1.96 (should equal ~0.975)
result, _ = quad(stats.norm.pdf, -np.inf, 1.96)
print(round(result, 4))

### 3.2 Solving ODEs

`solve_ivp` solves an initial value problem: dy/dt = f(t, y), y(t0) = y0.

In [ ]:
from scipy.integrate import solve_ivp

# simple decay: dy/dt = -0.5 * y, y(0) = 10
# exact solution: y(t) = 10 * exp(-0.5 * t)
def decay(t, y):
    return -0.5 * y

solution = solve_ivp(decay, t_span=(0, 10), y0=[10], dense_output=True)

t_eval = np.array([0, 2, 4, 6, 8, 10])
y_numerical = solution.sol(t_eval)[0].round(4)
y_exact     = (10 * np.exp(-0.5 * t_eval)).round(4)

print('t:          ', t_eval)
print('numerical:  ', y_numerical)
print('exact:      ', y_exact)

In [ ]:
# Lotka-Volterra (predator-prey): a system of two coupled ODEs
def lotka_volterra(t, state, alpha=1.0, beta=0.1, delta=0.075, gamma=1.5):
    x, y = state   # prey, predator
    dxdt = alpha * x - beta * x * y
    dydt = delta * x * y - gamma * y
    return [dxdt, dydt]

sol = solve_ivp(lotka_volterra, t_span=(0, 30), y0=[10, 5],
                t_eval=np.linspace(0, 30, 300))

print(f'prey range:     {sol.y[0].min():.1f} – {sol.y[0].max():.1f}')
print(f'predator range: {sol.y[1].min():.1f} – {sol.y[1].max():.1f}')

## 4. Interpolation (scipy.interpolate)

Ref: https://docs.scipy.org/doc/scipy/reference/interpolate.html

### 4.1 1D interpolation

Given a sparse set of known points, estimate values in between.

In [ ]:
from scipy.interpolate import interp1d

# sparse measurements
x_known = np.array([0, 1, 2, 3, 4, 5])
y_known = np.array([0, 1, 4, 2, 3, 1])

linear = interp1d(x_known, y_known, kind='linear')
cubic  = interp1d(x_known, y_known, kind='cubic')

x_new = np.array([0.5, 1.5, 2.5])
print('linear:', linear(x_new).round(4))
print('cubic: ', cubic(x_new).round(4))

### 4.2 Cubic splines

`CubicSpline` fits a smooth piecewise cubic curve that passes exactly through all data points.

In [ ]:
from scipy.interpolate import CubicSpline

cs = CubicSpline(x_known, y_known)

# evaluate the spline and its derivatives
x_dense = np.linspace(0, 5, 10)
print('values:      ', cs(x_dense).round(4))
print('derivatives: ', cs(x_dense, 1).round(4))  # first derivative

## 5. Linear Algebra (scipy.linalg)

`scipy.linalg` extends `numpy.linalg` with more algorithms and, in most cases, faster implementations.

Ref: https://docs.scipy.org/doc/scipy/reference/linalg.html

### 5.1 Solving a linear system

Solves Ax = b; faster and more numerically stable than computing the inverse.

In [ ]:
from scipy import linalg

A = np.array([[3, 1], [1, 2]], dtype=float)
b = np.array([9, 8], dtype=float)

x = linalg.solve(A, b)
print('solution:', x)              # x = [2, 3]
print('verify Ax=b:', A @ x)       # should equal b

### 5.2 Eigenvalues and eigenvectors

In [ ]:
A = np.array([[4, 2], [1, 3]], dtype=float)

values, vectors = linalg.eig(A)
print('eigenvalues: ', values.real)
print('eigenvectors:\n', vectors.real.round(4))

In [ ]:
# verify: A @ v = lambda * v
for i in range(len(values)):
    v = vectors[:, i]
    lhs = A @ v
    rhs = values[i].real * v
    print(f'v{i}: lhs={lhs.round(4)}, rhs={rhs.round(4)}')

### 5.3 Singular Value Decomposition

SVD decomposes any matrix into three matrices: A = U @ diag(s) @ Vt. It underpins PCA, least-squares solving, and data compression.

In [ ]:
A = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=float)

U, s, Vt = linalg.svd(A)
print('singular values:', s.round(4))
print('rank:', np.sum(s > 1e-10))  # number of non-zero singular values

In [ ]:
# reconstruct A from its SVD
S = np.diag(s)
A_reconstructed = U @ S @ Vt
print('max reconstruction error:', np.abs(A - A_reconstructed).max().round(10))

In [ ]:
# low-rank approximation: keep only the top k singular values
k = 1
A_approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
print('rank-1 approximation:\n', A_approx.round(2))